# 움직임 시험 3 — 영상 모델(Wan2.1 VACE 1.3B)로 블로킹 영상에 스타일 입히기 (Kaggle T4 권장, 학습 없음)
시험 1·2는 장마다 그림 모델로 다시 그렸다. 영상 모델은 장 사이를 함께 만들어 깜빡임이 적을 수 있다(GPT 2차 조사 경로, docs/GPT-RESEARCH-NAKAMURA-2026-10-26.md).
- 움직임: 블로킹 영상(num 15장, clip 12장)을 VACE '조절 영상'으로 넣고 마스크는 전부 흰색(전체를 새로 그림). 장 수는 모델 규칙(4n+1)에 맞춰 마지막 장을 반복해 17·13장.
- 화풍: 기본 영상 모델에는 나카무라 LoRA가 없다. 대신 3차 LoRA로 그린 폭발 그림 1장을 '참조 그림'으로 넣은 판과 안 넣은 판을 비교.
- 480×832, 30단계, guidance 5, 시드 1234. 결과는 원래 장 수만큼 잘라 24fps로 본다.
- T4는 bf16 연산이 느려(에뮬레이션) 시간이 더 걸릴 수 있다. 칸 2가 dtype을 고른다. 결과가 검게 나오면 그 사실을 알려 줄 것.
- 설정: **GPU T4 x2**, **Internet On** → Save & Run All. 모델 받기 포함 약 40–70분(추정). Colab은 램 12GB라 글 인코더(약 11GB)에서 멈출 수 있어 Kaggle 권장.
- 끝나면 `motion3_result.zip`(나란히 영상 `motion3.mp4` + 장별 그림)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 1) 패키지와 자료
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "accelerate>=1.0" ftfy imageio imageio-ffmpeg
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
!curl -sSL -o /tmp/ref.png "https://drive.usercontent.google.com/download?id=1pWgM5zWc6KK5-lBRmOs6zjovMZj-D-yG&export=download&confirm=t" && ls -la /tmp/ref.png


In [ ]:
# 2) Wan2.1 VACE 1.3B (diffusers). VAE는 fp32, 나머지는 bf16(T4는 에뮬레이션) — 안 되면 fp16
import torch
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
DT = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype", DT)
mid = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"
vae = AutoencoderKLWan.from_pretrained(mid, subfolder="vae", torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(mid, vae=vae, torch_dtype=DT)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()  # 글 인코더(큼)는 쓸 때만 GPU로


In [ ]:
# 3) 블로킹 × (참조 없음 / 참조 있음)
import numpy as np, json
from PIL import Image
P = ("2D hand-drawn anime effects animation, sakuga, a fiery explosion over a night city: a white-hot core with orange and red rims, "
     "layered cel shading with hard-edged color bands, sharp flame tips, embers and debris flying outward, no outlines, vivid colors, fixed camera")
N = ("blurry, low quality, 3D render, photorealistic, smooth gradients, grey smoke, text, watermark, static, overall gray, deformed")
H, W = 480, 832
ref = Image.open("/tmp/ref.png").convert("RGB").resize((W, H))
log = {}
for name in ("num", "clip"):
    blk = [Image.open(p).convert("RGB").resize((W, H)) for p in sorted(glob.glob(f"/tmp/blk_{name}/*.png"))]
    n0 = len(blk); n = ((n0 - 1 + 3) // 4) * 4 + 1  # 4n+1로 올림
    vid = blk + [blk[-1]] * (n - n0)
    mask = [Image.new("L", (W, H), 255)] * n
    for tag, refs in (("noref", None), ("ref", [ref])):
        g = torch.Generator("cpu").manual_seed(1234)
        frames = pipe(video=vid, mask=mask, reference_images=refs, prompt=P, negative_prompt=N, height=H, width=W,
                      num_frames=n, num_inference_steps=30, guidance_scale=5.0, generator=g, output_type="pil").frames[0]
        frames = frames[:n0]
        d = f"{OUT}/m3/{name}/{tag}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(frames):
            im.save(f"{d}/f{i:03d}.png")
        a = [np.asarray(f, np.float32) for f in frames]
        mean = float(np.mean([x.mean() for x in a]))
        print(name, tag, len(frames), "장, 평균 밝기", round(mean, 1), "(0에 가까우면 검은 그림 = dtype 문제)")
        log[f"{name} {tag}"] = round(mean, 1)
    d0 = f"{OUT}/m3/{name}/blocking"; os.makedirs(d0, exist_ok=True)
    for i, b in enumerate(blk):
        b.save(f"{d0}/f{i:03d}.png")
json.dump(log, open(f"{OUT}/motion3_log.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | 참조 없음 | 참조 있음. 24fps, 3번 반복
import cv2
rows = {n: {k: sorted(glob.glob(f"{OUT}/m3/{n}/{k}/*.png")) for k in ("blocking", "noref", "ref")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (448, 258)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion3.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1344, 516))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion3_result.zip m3 motion3.mp4 motion3_log.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion3_result.zip")
